[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/therestiscode/notebooks/blob/main/01-tensor-vs-ndarray/tensor.ipynb)

# What a tensor actually is

### One buffer, two names, and the one field that lets a neural network learn

**The Rest is Code** · [@therestiscode](https://youtube.com/@therestiscode)

---

This is the companion notebook to the short. The short is 46 seconds and it
makes one claim:

> **An array holds numbers. A tensor holds numbers — and how it got them.**
> "How it got them" is the entire reason a neural network can be trained.

Every assertion in the video is a cell down here that you can run. Where the
video had to move on, this stops and shows you the rest — including the answer
to the question the short deliberately ends without answering.

Run it top to bottom. **No GPU needed.** The accelerator section detects what
this machine actually has — `mps`, `cuda`, or neither — and says which, instead
of pretending.

```
pip install numpy torch
```

| | Section | In the short? |
|---|---|---|
| **1** | Two objects. One buffer. | the hook |
| **2** | Same answer. One of them keeps a receipt. | the turn |
| **3** | Cashing the receipt: `.backward()` | the payoff |
| **4** | The question the short ends on: `.device` | the question is. the answer is only here. |
| **5** | Buffer, shape, strides | cut for time — it lives here now |
| **6** | The whole difference, in one cell | — |

Sections 4 and 5 are the reason this notebook exists. Everything else is the
video with the pause button working.

In [1]:
import numpy as np
import torch

print(f"numpy {np.__version__}   torch {torch.__version__}")

# Whatever accelerator this machine actually has. `None` is fine — section 4
# says what it would have shown instead of pretending.
DEV = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else None
)
print(f"accelerator: {DEV or 'none — CPU only, section 4 will say so'}")

numpy 2.5.3   torch 2.14.0
accelerator: mps


---

## 1 · Two objects. One buffer.

`torch.from_numpy` does not copy. It wraps. There is one allocation on the heap
and two Python objects pointing at it.

In [2]:
a = np.arange(12, dtype=np.float32).reshape(3, 4)
t = torch.from_numpy(a)

print(a)
print(t)

[[ 0.  1.  2.  3.]
 [ 4.  5.  6.  7.]
 [ 8.  9. 10. 11.]]
tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])


Two different reprs, two different libraries. Now ask them both where they live.

In [3]:
np_ptr = a.__array_interface__["data"][0]

print(f"a.__array_interface__ ptr : {np_ptr:#x}")
print(f"t.data_ptr()              : {t.data_ptr():#x}")
print(f"same address              : {np_ptr == t.data_ptr()}")
print(f"a.nbytes                  : {a.nbytes}")
print(f"t.untyped_storage()       : {t.untyped_storage().nbytes()} bytes")

a.__array_interface__ ptr : 0x78939d88a0
t.data_ptr()              : 0x78939d88a0
same address              : True
a.nbytes                  : 48
t.untyped_storage()       : 48 bytes


Same address, same 48 bytes. Not a copy that happens to hold equal values —
**the same memory.**

> The addresses you get will not match the ones printed here, and that is
> correct: the OS hands out a different base address every run. What reproduces
> is that the two are **equal**.

So a write through one name is a write through the other.

In [4]:
a[0, 0] = 99.0
print("wrote a[0, 0] = 99.0   ->   t[0, 0] =", t[0, 0].item())

wrote a[0, 0] = 99.0   ->   t[0, 0] = 99.0


And it goes the other way too, which the short had no room for. This is one
buffer with two owners, not a one-directional view.

In [5]:
t[1, 1] = -5.0
print("wrote t[1, 1] = -5.0   ->   a[1, 1] =", a[1, 1])

wrote t[1, 1] = -5.0   ->   a[1, 1] = -5.0


> **The one place this bites in real code.** Because there is no copy, feeding a
> NumPy array into `torch.from_numpy` and then mutating the array underneath
> your model is a real class of bug. `torch.tensor(a)` copies;
> `torch.from_numpy(a)` does not. Pick deliberately.

In [6]:
copied = torch.tensor(a)          # copies
wrapped = torch.from_numpy(a)     # shares

print("torch.tensor(a)      shares memory:", copied.data_ptr() == np_ptr)
print("torch.from_numpy(a)  shares memory:", wrapped.data_ptr() == np_ptr)

torch.tensor(a)      shares memory: False
torch.from_numpy(a)  shares memory: True


---

## 2 · Same answer. One of them keeps a receipt.

Cube it, add it up. Both libraries do the same arithmetic and get the same
number, so the number is not where the difference lives.

In [7]:
n = np.array([2.0, 3.0])
x = torch.tensor([2.0, 3.0], requires_grad=True)

numpy_answer = (n ** 3).sum()
y = (x ** 3).sum()

print(f"numpy  (n ** 3).sum() : {numpy_answer}")
print(f"torch  (x ** 3).sum() : {y.item()}")
print(f"same answer           : {numpy_answer == y.item()}    (2³ + 3³ = 8 + 27)")

numpy  (n ** 3).sum() : 35.0
torch  (x ** 3).sum() : 35.0
same answer           : True    (2³ + 3³ = 8 + 27)


**NumPy hands you the number. PyTorch hands you the number — and a receipt.**

The receipt is a field on the result: `grad_fn`, the operation that produced it.
`x` itself has no `grad_fn`, because nothing produced `x` — we typed it.

In [8]:
print("x.grad_fn            :", x.grad_fn, "  <- leaf: nothing made it")
print("y.grad_fn            :", y.grad_fn)
print("x.requires_grad      :", x.requires_grad)
print("ndarray has grad_fn  :", hasattr(numpy_answer, "grad_fn"))

x.grad_fn            : None   <- leaf: nothing made it
y.grad_fn            : <SumBackward0 object at 0x1142708e0>
x.requires_grad      : True
ndarray has grad_fn  : False


`<SumBackward0>` is not a label. It is a node in a real graph, and the graph is
a data structure hanging off the tensor — every step, in order. You can walk it.

In [9]:
def walk(fn, depth=0):
    if fn is None:
        return
    print("  " * depth + type(fn).__name__)
    for nxt, _ in getattr(fn, "next_functions", ()):
        walk(nxt, depth + 1)

walk(y.grad_fn)

SumBackward0
  PowBackward0
    AccumulateGrad


Read it bottom-up and it is exactly what we typed: take `x` (`AccumulateGrad`,
the leaf), raise it to a power (`PowBackward0`), sum it (`SumBackward0`).

That is the itemised receipt. Nothing has been differentiated yet — the forward
pass just wrote down what it did on the way through.

---

## 3 · Cashing the receipt: `.backward()`

`.backward()` walks that structure in reverse, applying the chain rule at each
node, and leaves the result on `x.grad`.

In [10]:
y.backward()

print("x.grad         :", x.grad.tolist())
print("3x² by hand    :", (3 * x.detach() ** 2).tolist())
print("match          :", torch.allclose(x.grad, 3 * x.detach() ** 2))

x.grad         : [12.0, 27.0]
3x² by hand    : [12.0, 27.0]
match          : True


d/dx of x³ is 3x². At x = 2 that is 12; at x = 3 it is 27. Nobody told PyTorch
the derivative of a cube — it composed it out of the graph recorded while the
forward pass ran.

### What the number 12 actually means

"Nudge that first number a little and the answer moves twelve times as much."
That is a claim you can test without doing any calculus at all: move `x[0]` by a
hair, and see how far the answer moved.

In [11]:
def f(v):
    return (v ** 3).sum().item()

# float64 here purely so the measurement noise stays smaller than the answer.
base = torch.tensor([2.0, 3.0], dtype=torch.float64)
eps = 1e-4
nudge = torch.tensor([eps, 0.0], dtype=torch.float64)
moved = (f(base + nudge) - f(base)) / eps

print(f"gradient said x[0] moves the answer  : {x.grad[0].item()}x")
print(f"nudging x[0] by {eps} actually moved it : {moved:.4f}x")

gradient said x[0] moves the answer  : 12.0x
nudging x[0] by 0.0001 actually moved it : 12.0006x


So a gradient is two pieces of practical advice per number: **which way to push
it, and how hard.** That is all it is.

And that is the whole loop. Point the numbers downhill, take a step, do it
again — a few billion times, and you have trained a neural network. Here it is
at its smallest: two parameters learning the line `y = 3x + 2`, from nothing but
`.backward()` and subtraction.

In [12]:
xs = torch.linspace(-1.0, 1.0, 64)
ys = 3.0 * xs + 2.0                       # the line we are asking it to find

w = torch.zeros(1, requires_grad=True)    # it starts out knowing nothing
b = torch.zeros(1, requires_grad=True)

for step in range(301):
    loss = ((w * xs + b - ys) ** 2).mean()  # how wrong we are
    loss.backward()                         # read the receipt
    with torch.no_grad():
        w -= 0.1 * w.grad                   # step downhill
        b -= 0.1 * b.grad
        w.grad.zero_()
        b.grad.zero_()
    if step % 60 == 0:
        print(f"step {step:>3}   loss {loss.item():9.2e}   w {w.item():.3f}   b {b.item():.3f}")

print(f"\nlearned  w = {w.item():.3f}   b = {b.item():.3f}      (we asked for 3 and 2)")

step   0   loss  7.10e+00   w 0.206   b 0.400
step  60   loss  5.98e-04   w 2.961   b 2.000
step 120   loss  1.16e-07   w 2.999   b 2.000
step 180   loss  2.27e-11   w 3.000   b 2.000
step 240   loss  1.01e-12   w 3.000   b 2.000
step 300   loss  1.01e-12   w 3.000   b 2.000

learned  w = 3.000   b = 2.000      (we asked for 3 and 2)


Nothing in that loop knew any calculus. Each step is a receipt, read backwards.

An `ndarray` has no answer to any of this, because it kept no record of anything.

In [13]:
try:
    np.array([2.0, 3.0]).backward()
except AttributeError as e:
    print("ndarray.backward() ->", type(e).__name__, "-", e)

ndarray.backward() -> AttributeError - 'numpy.ndarray' object has no attribute 'backward'


---

## 4 · The question the short ends on: `.device`

> *"Move this tensor to your GPU — what happens to the memory they were
> sharing?"*

Here is the answer. But first, the correction, because the tidy version of this
beat — "tensors have a device, arrays don't" — is false.

In [14]:
print("np.ndarray has .device :", hasattr(a, "device"))
print("a.device               :", repr(a.device))
print("t.device               :", t.device)

np.ndarray has .device : True
a.device               : 'cpu'
t.device               : cpu


NumPy 2 exposes `.device` as part of the
[Python array API standard](https://data-apis.org/array-api/). It returns
`'cpu'`. It will always return `'cpu'`.

The real difference is not the attribute. It is that **NumPy has nowhere to go.**

In [15]:
try:
    a.to("mps")
except AttributeError as e:
    print("a.to('mps')  ->", type(e).__name__, "-", e)

a.to('mps')  -> AttributeError - 'numpy.ndarray' object has no attribute 'to'


**NumPy's `.device` is a label. PyTorch's is a destination.**

And moving somewhere real means a new allocation on a different chip — so the
address changes, and the buffer this tensor shared with the array back in
section 1 is not shared any more.

In [16]:
if DEV is None:
    print("No accelerator on this machine, so there is nowhere to move to.")
    print("On Apple silicon DEV would be 'mps'; on an NVIDIA box, 'cuda'.")
    print("The assertion this cell makes is: g.data_ptr() != t.data_ptr().")
else:
    g = t.to(DEV)
    print(f"t.device            : {t.device}")
    print(f"g.device            : {g.device}")
    print(f"t.data_ptr()        : {t.data_ptr():#x}")
    print(f"g.data_ptr()        : {g.data_ptr():#x}")
    print(f"still numpy buffer  : {g.data_ptr() == np_ptr}")

t.device            : cpu
g.device            : mps:0
t.data_ptr()        : 0x78939d88a0
g.data_ptr()        : 0x788e62ca80
still numpy buffer  : False


`False` — and that is the answer to the closing question. The interesting part
is not that the pointer changed, it is **how quietly the link came off.** Write
to the array now. `t` still sees it. `g` never will again.

In [17]:
a[0, 0] = 1234.0

print(f"a[0, 0]                 : {a[0, 0]}")
print(f"t[0, 0]  (still shared) : {t[0, 0].item()}")
if DEV is None:
    print("g[0, 0]  (no accelerator on this machine — it would still read 99.0)")
else:
    print(f"g[0, 0]  ({DEV} copy)      : {g[0, 0].item()}   <- stale, and nothing warned you")

a[0, 0]                 : 1234.0
t[0, 0]  (still shared) : 1234.0
g[0, 0]  (mps copy)      : 99.0   <- stale, and nothing warned you


No exception, no warning. The GPU tensor is a snapshot taken at the moment of
the copy, and it has been quietly wrong ever since.

> `.to()` is a **copy**, not a move — `t` is still sitting in system memory
> afterwards. That is also why `g.numpy()` raises: NumPy cannot read GPU memory,
> so you have to come back through the CPU first.

In [18]:
if DEV is not None:
    try:
        g.numpy()
    except (TypeError, RuntimeError) as e:
        print(f"g.numpy() -> {type(e).__name__}: {str(e).splitlines()[0]}")
    print("g.cpu().numpy() works  :", type(g.cpu().numpy()).__name__)
else:
    print("skipped — no accelerator")

g.numpy() -> TypeError: can't convert mps:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.
g.cpu().numpy() works  : ndarray


---

## 5 · The part the short cut: buffer, shape, strides

This was in the script and it got cut for time. It is also the best thing in
this notebook if you like knowing how the machine works, so it gets to live
here, where the reader who wants it is already committed.

Strip either object down and you get three things, and they agree on the first
two exactly.

In [19]:
print(f"a.shape    : {a.shape}        t.shape    : {tuple(t.shape)}")
print(f"a.dtype    : {a.dtype}    t.dtype    : {t.dtype}")
print(f"a.itemsize : {a.itemsize} bytes")

a.shape    : (3, 4)        t.shape    : (3, 4)
a.dtype    : float32    t.dtype    : torch.float32
a.itemsize : 4 bytes


### The third one differs — and only in its unit

**NumPy measures strides in bytes. PyTorch measures them in elements.** Same
layout, two unit systems.

In [20]:
print(f"a.strides   : {a.strides}   <- BYTES")
print(f"t.stride()  : {t.stride()}     <- ELEMENTS")
print()
print(f"a.strides converted to elements : {tuple(s // a.itemsize for s in a.strides)}")
print(f"t.stride()  converted to bytes  : {tuple(s * a.itemsize for s in t.stride())}")

a.strides   : (16, 4)   <- BYTES
t.stride()  : (4, 1)     <- ELEMENTS

a.strides converted to elements : (4, 1)
t.stride()  converted to bytes  : (16, 4)


A stride is just "how far to step to reach the next one along this axis". To get
to the next **row** of a `(3, 4)` float32 array you skip 4 elements — which is
16 bytes. Both libraries are saying the identical thing.

### Why this matters: a view costs nothing

Because shape and strides are *interpretation* laid over a flat buffer, the
operations that only change them are free. No data moves.

In [21]:
print("original   shape/strides:", a.shape, a.strides)
print("transposed shape/strides:", a.T.shape, a.T.strides)
print("reshaped   shape/strides:", a.reshape(2, 6).shape, a.reshape(2, 6).strides)
print()
print("a.T is the same buffer        :", a.T.__array_interface__["data"][0] == np_ptr)
print("a.reshape(2,6) same buffer    :", a.reshape(2, 6).__array_interface__["data"][0] == np_ptr)
print()
print("t.T stride()                  :", t.T.stride())
print("t.T is the same buffer        :", t.T.data_ptr() == np_ptr)

original   shape/strides: (3, 4) (16, 4)
transposed shape/strides: (4, 3) (4, 16)
reshaped   shape/strides: (2, 6) (24, 4)

a.T is the same buffer        : True
a.reshape(2,6) same buffer    : True

t.T stride()                  : (1, 4)
t.T is the same buffer        : True


The strides reversed. Nothing was rearranged in memory. That is the whole trick,
and it is identical in both libraries — which is the point: **everything in this
section is true of both objects.** Down to the address.

---

## 6 · The whole difference, in one cell

Everything a `torch.Tensor` has that a `numpy.ndarray` does not:

In [22]:
candidates = ("grad_fn", "requires_grad", "backward", "grad", "to", "device", "shape", "dtype")
only_tensor = [n for n in candidates if hasattr(t, n) and not hasattr(a, n)]
both = [n for n in candidates if hasattr(t, n) and hasattr(a, n)]

print("on Tensor, NOT on ndarray :", only_tensor)
print("on both                   :", both)

on Tensor, NOT on ndarray : ['grad_fn', 'requires_grad', 'backward', 'grad', 'to']
on both                   : ['device', 'shape', 'dtype']


Note what is **not** in the first list: `device`. And note that four of the five
entries that are — `grad_fn`, `requires_grad`, `backward`, `grad` — are one
single feature wearing four names.

So the honest summary is the one the short ends on:

> **An array holds numbers.
> A tensor holds numbers, and how it got them.**

That second half is the entire mechanism deep learning runs on. Every parameter
update in every model you have ever used is `.backward()` walking a graph that a
tensor recorded about itself — the same three lines as the toy loop in section 3,
with more numbers and more patience.

---

## Try it yourself

Predict first, then run. The prediction is where the learning is.

1. **Break the sharing.** Find three operations that return a tensor sharing
   `a`'s buffer, and three that must copy. (Check `.data_ptr()`. Start with
   `t[1:]`, `t.T`, `t.reshape(...)`, `t.contiguous()`, `t + 0`, `t.clone()`.)
2. **Non-contiguous strides.** What is `a[:, ::2].strides`? Explain the first
   number without running it, then check.
3. **A deeper graph.** Build `z = (x ** 2 + 3 * x).sum()`, predict `x.grad` by
   hand, then verify. Walk `z.grad_fn` and match each node to a term you wrote.
4. **The nudge, at a different point.** Section 3 measured the slope at x = 2 and
   got 12. Do it at x = 5 before you run it — then check with the same
   finite-difference cell.
5. **`requires_grad` mid-stream.** What happens to `grad_fn` if you call
   `.detach()` halfway through a chain? Why is that the mechanism behind
   `torch.no_grad()` — and what does it buy you at inference time?
6. **Round trip.** Get the trained `w` from section 3 back into a NumPy array.
   Why does `w.numpy()` complain, and what is the fix telling you about the
   receipt? (Two of them work: `.detach()` and `torch.no_grad()`.)

---

### The Rest is Code

Short videos about what the code is actually doing — [@therestiscode](https://youtube.com/@therestiscode).

This notebook is the long version of a 46-second one. Fork it, run it, break it,
teach from it. If a cell disagrees with the video, trust the cell — it ran.